# Credit Risk Prediction

## Objective
Build a machine learning model to predict whether a customer will default on a loan.

## Problem Type
Binary Classification  
Target Variable: Risk (1 = Default, 0 = Non-Default)

## Business Impact
Accurate default prediction helps:
- Reduce financial losses
- Improve loan approval decisions
- Optimize interest rates


In [ ]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")

In [ ]:
column_names = [
    "Status", "Duration", "CreditHistory", "Purpose",
    "CreditAmount", "Savings", "EmploymentDuration",
    "InstallmentRate", "PersonalStatusSex", "OtherDebtors",
    "ResidenceDuration", "Property", "Age",
    "OtherInstallmentPlans", "Housing", "ExistingCredits",
    "Job", "NumberOfDependents", "Telephone",
    "ForeignWorker", "Risk"
]

df = pd.read_csv(
    "german.data",
    sep=" ",
    header=None,
    names=column_names
)

df.head()

In [ ]:
df["Risk"] = df["Risk"].map({1: 0, 2: 1})
df["Risk"].value_counts()

In [ ]:
df.info()

In [ ]:
y = df["Risk"]
X = df.drop(columns=["Risk"])

X_encoded = pd.get_dummies(X, drop_first=True)

print("Original shape:", X.shape)
print("Encoded shape:", X_encoded.shape)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


In [ ]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]


In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
threshold = 0.3

y_pred_adjusted = (y_prob >= threshold).astype(int)

print("New Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_adjusted))

print("\nNew Classification Report:")
print(classification_report(y_test, y_pred_adjusted))

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_encoded.columns)
top_features = importances.sort_values(ascending=False).head(10)

plt.figure(figsize=(8,5))
top_features.sort_values().plot(kind='barh')
plt.title("Top 10 Feature Importances")
plt.show()

top_features

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)
lr_model.fit(X_train, y_train)

lr_pred = lr_model.predict(X_test)
lr_prob = lr_model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, lr_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr_prob))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, lr_pred))
print("\nClassification Report:")
print(classification_report(y_test, lr_pred))

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    use_label_encoder=False
)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)
xgb_prob = xgb_model.predict_proba(X_test)[:, 1]

print("XGBoost Performance")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb_prob))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_pred))
print("\nClassification Report:")
print(classification_report(y_test, xgb_pred))

In [ ]:
rf_auc = roc_auc_score(y_test, y_prob)
lr_auc = roc_auc_score(y_test, lr_prob)
xgb_auc = roc_auc_score(y_test, xgb_prob)

print("Model Comparison (ROC-AUC)")
print("Random Forest:", rf_auc)
print("Logistic Regression:", lr_auc)
print("XGBoost:", xgb_auc)

auc_scores = {
    "Random Forest": rf_auc,
    "Logistic Regression": lr_auc,
    "XGBoost": xgb_auc
}

best_model_name = max(auc_scores, key=auc_scores.get)

if best_model_name == "Random Forest":
    best_model = model
elif best_model_name == "Logistic Regression":
    best_model = lr_model
else:
    best_model = xgb_model

print("\nBest Model Selected:", best_model_name)

In [ ]:
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
from sklearn.model_selection import cross_val_score

rf_cv_scores = cross_val_score(
    model,
    X_encoded,
    y,
    cv=cv,
    scoring="roc_auc"
)

print("Random Forest CV ROC-AUC Scores:", rf_cv_scores)
print("Mean ROC-AUC:", rf_cv_scores.mean())
print("Std Dev:", rf_cv_scores.std())


In [ ]:
lr_cv_scores = cross_val_score(
    lr_model,
    X_encoded,
    y,
    cv=cv,
    scoring="roc_auc"
)

print("Logistic Regression CV ROC-AUC Scores:", lr_cv_scores)
print("Mean ROC-AUC:", lr_cv_scores.mean())
print("Std Dev:", lr_cv_scores.std())


In [ ]:
xgb_cv_scores = cross_val_score(
    xgb_model,
    X_encoded,
    y,
    cv=cv,
    scoring="roc_auc"
)

print("XGBoost CV ROC-AUC Scores:", xgb_cv_scores)
print("Mean ROC-AUC:", xgb_cv_scores.mean())
print("Std Dev:", xgb_cv_scores.std())


In [ ]:
cv_results_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "XGBoost"],
    "Mean ROC-AUC": [
        np.mean(lr_cv_scores),
        np.mean(rf_cv_scores),
        np.mean(xgb_cv_scores)
    ],
    "Std Dev": [
        np.std(lr_cv_scores),
        np.std(rf_cv_scores),
        np.std(xgb_cv_scores)
    ]
})

cv_results_df


In [ ]:
import shap

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)
shap.summary_plot(shap_values, X_test)


In [ ]:
coefficients = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": lr_model.coef_[0]
})

coefficients["Abs_Coefficient"] = np.abs(coefficients["Coefficient"])
coefficients_sorted = coefficients.sort_values(by="Abs_Coefficient", ascending=False)
coefficients_sorted.head(20)

In [ ]:
coefficients_sorted["Odds_Ratio"] = np.exp(coefficients_sorted["Coefficient"])
coefficients_sorted.head(20)

## Logistic Regression Interpretation

The logistic regression coefficients reveal key risk drivers. 
Higher installment burden and certain loan purposes significantly increase default probability. 
Conversely, strong savings history, favorable checking account status, and positive credit history substantially reduce risk. 
Odds ratios were computed to quantify the magnitude of impact on default likelihood.
